# Lab 6 Student Version (Google Colab)
## Evaluate a Feedforward Neural Network (MLP) on Fashion-MNIST with Full Metrics

This notebook is a **student-learning version** of the attached Lab 6 guide. It preserves **every topic and step** from the lab and adds short explanations of **what you are learning** and **why each step matters**.

## Lab overview
In this lab, you will **train and evaluate** a feedforward neural network (MLP) on the **Fashion-MNIST** dataset. You will:
- train the model,
- make predictions on the test set,
- compute evaluation metrics (accuracy, precision, recall, F1-score),
- and visualize a **confusion matrix** to understand class-level performance.

### In this lab, you will
- Load Fashion-MNIST using `torchvision`
- Define and train a simple MLP classifier
- Make predictions and evaluate using `sklearn` metrics
- Visualize a colorful confusion matrix using `seaborn`

### Estimated completion time
- **35 minutes**

### Runtime type
- Set runtime type to: **CPU**


# Task 0 (Colab setup): Imports and environment check

### What you are learning
Before running the lab, you verify your environment and imports. This reduces setup errors and makes your results easier to reproduce.


In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Runtime recommendation for this lab: CPU")


# Task 1: Loading Fashion-MNIST dataset using `torchvision.datasets`

### What you are learning
You are learning how to load a standard image classification dataset with `torchvision`, build `DataLoader`s for batching, and visualize samples with labels. Visual inspection is a great first sanity-check to confirm your data looks correct.

### Steps (from the lab)
1. Import necessary libraries.
2. Define transform to convert images to tensors.
3. Load the training and test sets.
4. Create data loaders.
5. Print dataset sizes.
6. Define class labels.
7. Visualize training samples clearly (helper function).
8. Show three random examples from the training set.
9. Review the output.


In [ ]:
# 1. Import necessary libraries.
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# 2. Define transform to convert images to tensors.
transform = transforms.ToTensor()

# 3. Load the training and test sets.
trainset = torchvision.datasets.FashionMNIST(root='./data',
                                            train=True, download=True, transform=transform)
testset = torchvision.datasets.FashionMNIST(root='./data',
                                           train=False, download=True, transform=transform)

# 4. Create data loaders.
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

# 5. Print dataset sizes.
print(f"Number of training samples: {len(trainset)}")
print(f"Number of test samples: {len(testset)}")

# 6. Define class labels.
classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [ ]:
# 7. Visualize training samples clearly.
def imshow(img_tensor, label):
    img = img_tensor.numpy().squeeze()
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(img, cmap='gray', interpolation='nearest')
    plt.title(f"Label: {classes[label]}", fontsize=10)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# 8. Show three random examples from the training set.
for i in range(3):
    img, label = trainset[i]
    imshow(img, label)


### Task 1 explanation (learning takeaways)
- `transforms.ToTensor()` converts images to PyTorch tensors (and scales pixel values into `[0, 1]`).
- `DataLoader` batches data and shuffles training data to improve learning.
- Visualizing a few samples helps you confirm:
  - the images are loaded correctly,
  - labels align with what you see,
  - and shapes look as expected.


# Task 2: Defining and training a simple MLP using `nn.Module`

### What you are learning
You are learning how to:
- define a classifier using `nn.Module` and `nn.Sequential`,
- train it with `CrossEntropyLoss` (standard for multi-class classification),
- optimize with Adam,
- and track loss and accuracy across epochs.

### Steps (from the lab)
1. Define the MLP model using `nn.Module`.
2. Initialize model, loss function, optimizer.
3. Run a training loop for 5 epochs and print loss + accuracy each epoch.
4. Review the output.


In [ ]:
# 1. Define the MLP model using nn.Module.
import torch.nn as nn
import torch.optim as optim

class FashionMLP(nn.Module):
    def __init__(self):
        super(FashionMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.model(x)

# 2. Initialize model, loss function, optimizer.
model = FashionMLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)


In [ ]:
# 3. Training loop.
epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in trainloader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = correct / total
    avg_loss = total_loss / len(trainloader)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.4f}")


### Task 2 explanation (learning takeaways)
- `CrossEntropyLoss` expects:
  - model outputs as raw scores (logits) of shape `(batch, 10)`
  - labels as integer class IDs `0–9`
- The training loop pattern is:
  1) forward pass → 2) compute loss → 3) backward pass → 4) optimizer step
- Accuracy is computed by comparing the predicted class (`argmax`) to the true label.


# Task 3: Making predictions on the test set

### What you are learning
You are learning how to:
- switch the model to evaluation mode (`model.eval()`),
- run inference without gradients (`torch.no_grad()`),
- store true labels and predicted labels for later metric computation,
- visualize a few predictions,
- then run predictions across the full test set to collect labels for evaluation.

### Steps (from the lab)
1. Set the model to evaluate mode.
2. Create storage for predictions.
3. Get one batch from the test set for visualization.
4. Run inference on sample batch.
5. Store for evaluation later.
6. Plot 6 sample images with predictions.
7. Continue prediction on full test set for metrics.
8. Review the outputs.


In [ ]:
# 1. Set the model to evaluate mode.
import matplotlib.pyplot as plt
model.eval()

# 2. Storage for all predictions.
true_labels = []
predicted_labels = []

# 3. Get one batch from the test set for visualization.
dataiter = iter(testloader)
images, labels = next(dataiter)

# 4. Run inference on sample batch.
with torch.no_grad():
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)

# 5. Store for evaluation later.
true_labels.extend(labels.numpy())
predicted_labels.extend(predicted.numpy())


In [ ]:
# 6. Plot 6 sample images with predictions.
plt.figure(figsize=(12, 6))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    img = images[i].permute(1, 2, 0).numpy()
    plt.imshow(img.squeeze(), cmap='gray')
    plt.title(f"True: {labels[i].item()} | Pred: {predicted[i].item()}")
    plt.axis('off')

plt.suptitle(" Sample Predictions from Test Set")
plt.tight_layout()
plt.show()


In [ ]:
# 7. (Continue prediction on full test set for metrics.)
# NOTE: This lab intentionally keeps the earlier sample-batch predictions in the lists,
# so the final count becomes 10000 + 64 = 10064 (matching the lab output).
with torch.no_grad():
    for inputs, labels in testloader:
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        true_labels.extend(labels.numpy())
        predicted_labels.extend(preds.numpy())

print(f" Total Predictions Collected: {len(predicted_labels)}")


### Task 3 explanation (learning takeaways)
- `model.eval()` turns off training-only behavior (important for dropout/batchnorm models).
- `torch.no_grad()` saves memory and speeds up inference by skipping gradient tracking.
- Storing `true_labels` and `predicted_labels` lets you compute metrics like precision/recall/F1.
- Plotting sample predictions helps you sanity-check that the model is making reasonable guesses.


# Task 4: Evaluating the model using accuracy, report, and confusion matrix

### What you are learning
You are learning how to evaluate a classifier beyond “overall accuracy” using:
- **classification report** (precision, recall, F1-score per class),
- and a **confusion matrix** (raw counts of true vs predicted classes).

These tools help you see where the model performs well and where it gets confused.

### Steps (from the lab)
1. Import sklearn evaluation utilities.
2. Calculate accuracy.
3. Print classification report (precision/recall/F1).
4. Print confusion matrix (raw counts).
5. Review the output.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Calculate accuracy.
accuracy = accuracy_score(true_labels, predicted_labels)
print(f" Overall Test Accuracy: {accuracy:.4f}")

# 2. Print the classification report: precision, recall, F1.
print("\n Classification Report:")
print(classification_report(true_labels, predicted_labels, target_names=classes))

# 3. Print the confusion matrix.
cm = confusion_matrix(true_labels, predicted_labels)
print("\n Confusion Matrix (raw counts):")
print(cm)


### Task 4 explanation (learning takeaways)
- **Accuracy** answers: “What fraction of predictions are correct overall?”
- **Precision** answers: “When the model predicts a class, how often is it right?”
- **Recall** answers: “Of the true items in a class, how many did the model find?”
- **F1-score** balances precision and recall.
- The **confusion matrix** shows which classes are commonly mistaken for others.


# Task 5: Visualizing the confusion matrix using seaborn

### What you are learning
You are learning how to convert the confusion matrix into an easy-to-read **heatmap** with class names. This makes class-level performance patterns clear and helps you spot systematic confusions.

### Steps (from the lab)
1. Import necessary libraries.
2. Compute confusion matrix (if not already available).
3. Set figure size.
4. Plot confusion matrix using seaborn heatmap.
5. Add labels and titles.
6. Review the output.


In [ ]:
# 1. Import necessary libraries.
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

# 2. Compute the confusion matrix again (if not already available).
cm = confusion_matrix(true_labels, predicted_labels)

# 3. Set figure size.
plt.figure(figsize=(10, 8))

# 4. Plot confusion matrix using seaborn heatmap.
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes,
            linewidths=0.5, linecolor='gray')

# 5. Add labels and titles.
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - Fashion-MNIST', fontsize=14)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


### Task 5 explanation (learning takeaways)
- The heatmap makes it easy to spot:
  - strong diagonal values (correct predictions),
  - off-diagonal “hot spots” where the model confuses two classes.
- This is especially helpful when you want to improve a model, because it points to specific class pairs to investigate.


# Lab review

1. A confusion matrix helps visualize how well a classifier performs on each class.  
A. True  
B. False  

2. Which of the following metrics is NOT a part of the classification report?  
A. Precision  
B. Recall  
C. F1-Score  
D. Mean Squared Error  

---

## STOP
You have successfully completed this lab.


<details>
<summary><strong>Optional self-check answers (click to expand)</strong></summary>

1. **A (True)**  
2. **D (Mean Squared Error)**

</details>
